# Validacion walk-forward (5 origenes temporales, distintos regimenes hidrologicos)

Tarea de ruta critica prometida en el Anexo 1 (Fase 2, OE2): "verificando la estabilidad de los
hiperparametros elegidos en 5 origenes temporales que abarcan distintos regimenes hidrologicos".

En vez de un unico corte train/test (2019-2025 / 2026), se repite el entrenamiento y evaluacion en
5 origenes distintos, cada uno cayendo en un regimen ENSO diferente segun data/external/oni_index.csv.
Cada origen es un escenario "ventana creciente": se entrena con todo lo disponible hasta el corte, y
se evalua en el trimestre inmediatamente posterior -- asi se simula la situacion real de re-entrenar
periodicamente con el historial que se tiene en ese momento.

Se reutilizan las mismas features ya construidas en 05_features_compartidas_juan.ipynb
(dataset_features_2019_2025.csv + dataset_features_2026.csv), y la misma configuracion final
de XGBoost (06_modelo_xgboost_juan.ipynb) y Prophet (04_modelo_prophet_juan.ipynb) -- aqui no se
vuelve a tunear nada, solo se mide que tan estable es lo ya elegido.

In [1]:
# --- Celda de arranque ---
import pandas as pd
import numpy as np
from pathlib import Path

def encontrar_raiz_proyecto(marcador="requirements.txt"):
    actual = Path.cwd()
    for carpeta in [actual, *actual.parents]:
        if (carpeta / marcador).exists():
            return carpeta
    raise FileNotFoundError(f"No encontre '{marcador}' subiendo desde {actual}")

RAIZ = encontrar_raiz_proyecto()
print("Raiz del proyecto:", RAIZ)

def calcular_metricas(y_real, y_pred):
    y_real, y_pred = np.asarray(y_real, dtype=float), np.asarray(y_pred, dtype=float)
    error = y_real - y_pred
    mae = np.abs(error).mean()
    rmse = np.sqrt((error ** 2).mean())
    mape = (np.abs(error) / y_real).mean() * 100
    return mae, rmse, mape


Raiz del proyecto: C:\Users\mgdbj\xm-spot-price-predictor


In [2]:
# --- Cargar el dataset de features ya construido (mismo insumo que 04 y 06) ---
df_train = pd.read_csv(RAIZ / "data" / "processed" / "dataset_features_2019_2025.csv", parse_dates=["fecha_hora"])
df_test = pd.read_csv(RAIZ / "data" / "processed" / "dataset_features_2026.csv", parse_dates=["fecha_hora"])

df_completo = pd.concat([df_train, df_test], ignore_index=True).sort_values("fecha_hora").reset_index(drop=True)
print("Rango disponible:", df_completo["fecha_hora"].min(), "a", df_completo["fecha_hora"].max())
print("Filas:", df_completo.shape[0])


Rango disponible: 2019-01-31 23:00:00 a 2026-08-05 23:00:00
Filas: 65833


In [3]:
# --- Los 5 origenes: ventana creciente, cada uno cae en un regimen ENSO distinto ---
# Regimenes segun data/external/oni_index.csv (ONI real, no proyectado):
#   2020 Jul-Sep: -0.3 a -0.8  (La Nina, inicio)
#   2021 Jul-Sep: -0.3 a -0.6  (La Nina, continuacion)
#   2022 Oct-Dic: -0.9 a -0.7  (La Nina "triple-dip", el mas frio de la serie)
#   2023 Oct-Dic:  1.7 a  2.0  (El Nino fuerte, construyendose)
#   2024 Ene-Mar:  1.8 a  1.2  (El Nino en su pico, ya bajando)
origenes = [
    {"nombre": "Origen 1", "regimen": "La Nina (inicio)",       "corte_train": "2020-07-01", "test_inicio": "2020-07-01", "test_fin": "2020-09-30"},
    {"nombre": "Origen 2", "regimen": "La Nina (continuacion)", "corte_train": "2021-07-01", "test_inicio": "2021-07-01", "test_fin": "2021-09-30"},
    {"nombre": "Origen 3", "regimen": "La Nina (triple-dip)",   "corte_train": "2022-10-01", "test_inicio": "2022-10-01", "test_fin": "2022-12-31"},
    {"nombre": "Origen 4", "regimen": "El Nino (fuerte)",       "corte_train": "2023-10-01", "test_inicio": "2023-10-01", "test_fin": "2023-12-31"},
    {"nombre": "Origen 5", "regimen": "El Nino (pico)",         "corte_train": "2024-01-01", "test_inicio": "2024-01-01", "test_fin": "2024-03-31"},
]

for o in origenes:
    ventana = df_completo[(df_completo["fecha_hora"] >= o["test_inicio"]) & (df_completo["fecha_hora"] <= o["test_fin"])]
    o["oni_min"] = ventana["oni"].min()
    o["oni_max"] = ventana["oni"].max()
    o["n_train"] = (df_completo["fecha_hora"] < o["corte_train"]).sum()
    o["n_test"] = len(ventana)
    print(f"{o['nombre']:10s} [{o['regimen']:22s}] train<{o['corte_train']}  test {o['test_inicio']}..{o['test_fin']}  "
          f"ONI [{o['oni_min']:.1f}, {o['oni_max']:.1f}]  n_train={o['n_train']}  n_test={o['n_test']}")


Origen 1   [La Nina (inicio)      ] train<2020-07-01  test 2020-07-01..2020-09-30  ONI [-0.8, -0.3]  n_train=12385  n_test=2185
Origen 2   [La Nina (continuacion)] train<2021-07-01  test 2021-07-01..2021-09-30  ONI [-0.6, -0.3]  n_train=21145  n_test=2185
Origen 3   [La Nina (triple-dip)  ] train<2022-10-01  test 2022-10-01..2022-12-31  ONI [-0.9, -0.7]  n_train=32113  n_test=2185
Origen 4   [El Nino (fuerte)      ] train<2023-10-01  test 2023-10-01..2023-12-31  ONI [1.7, 2.0]  n_train=40873  n_test=2185
Origen 5   [El Nino (pico)        ] train<2024-01-01  test 2024-01-01..2024-03-31  ONI [1.2, 1.8]  n_train=43081  n_test=2161


In [4]:
# --- Mismas features que 06_modelo_xgboost_juan.ipynb (exclusion identica) ---
columnas_excluir = ["fecha_hora", "precio_bolsa", "demanda", "generacion",
                     "anio", "mes", "hora", "dia_semana", "dia_anio"]
columnas_features = [c for c in df_completo.columns if c not in columnas_excluir]

# Mismos regresores que 04_modelo_prophet_juan.ipynb (version final corregida)
columnas_regresoras_prophet = ["aportes_hidricos", "volumen_embalses", "oni", "es_pandemia", "precio_lag24h"]

print(f"{len(columnas_features)} features para XGBoost.")
print(f"{len(columnas_regresoras_prophet)} regresores para Prophet.")


33 features para XGBoost.
5 regresores para Prophet.


In [5]:
# --- Walk-forward: XGBoost (misma config final: depth=3, lr=0.01) + persistencia ---
import xgboost as xgb

resultados = []

for o in origenes:
    train = df_completo[df_completo["fecha_hora"] < o["corte_train"]].dropna(subset=columnas_features)
    test = df_completo[(df_completo["fecha_hora"] >= o["test_inicio"]) & (df_completo["fecha_hora"] <= o["test_fin"])]

    X_train, y_train = train[columnas_features], np.log(train["precio_bolsa"])
    X_test, y_test = test[columnas_features], test["precio_bolsa"]

    modelo = xgb.XGBRegressor(n_estimators=500, max_depth=3, learning_rate=0.01,
                               subsample=0.8, colsample_bytree=0.8, random_state=42)
    modelo.fit(X_train, y_train)
    y_pred = np.exp(modelo.predict(X_test))

    mae, rmse, mape = calcular_metricas(y_test, y_pred)
    resultados.append({"origen": o["nombre"], "regimen": o["regimen"], "modelo": "XGBoost",
                        "mae": mae, "rmse": rmse, "mape": mape})

    # Persistencia: precio_lag24h ya es exactamente eso (precio_bolsa desplazado 24h)
    mae_p, rmse_p, mape_p = calcular_metricas(test["precio_bolsa"], test["precio_lag24h"])
    resultados.append({"origen": o["nombre"], "regimen": o["regimen"], "modelo": "Persistencia",
                        "mae": mae_p, "rmse": rmse_p, "mape": mape_p})

    print(f"{o['nombre']} [{o['regimen']}]  XGBoost MAE {mae:6.2f} RMSE {rmse:6.2f} MAPE {mape:5.2f}%   |  "
          f"Persistencia MAE {mae_p:6.2f} RMSE {rmse_p:6.2f} MAPE {mape_p:5.2f}%")


Origen 1 [La Nina (inicio)]  XGBoost MAE  16.02 RMSE  21.30 MAPE 10.86%   |  Persistencia MAE  15.71 RMSE  22.85 MAPE 10.93%


Origen 2 [La Nina (continuacion)]  XGBoost MAE   8.06 RMSE  12.94 MAPE  7.80%   |  Persistencia MAE   6.32 RMSE  14.57 MAPE  5.60%


Origen 3 [La Nina (triple-dip)]  XGBoost MAE  42.25 RMSE  66.63 MAPE 15.85%   |  Persistencia MAE  36.07 RMSE  71.48 MAPE 15.02%


Origen 4 [El Nino (fuerte)]  XGBoost MAE 129.97 RMSE 177.40 MAPE 23.81%   |  Persistencia MAE  90.42 RMSE 142.26 MAPE 19.58%


Origen 5 [El Nino (pico)]  XGBoost MAE  46.96 RMSE  70.19 MAPE  9.86%   |  Persistencia MAE  42.49 RMSE  71.56 MAPE  9.28%


In [6]:
# --- Walk-forward: Prophet (misma config final: cps=0.005, mismos regresores) ---
from prophet import Prophet
import logging
logging.getLogger("cmdstanpy").setLevel(logging.WARNING)  # silenciar el log de cada fit

for o in origenes:
    train = df_completo[df_completo["fecha_hora"] < o["corte_train"]].dropna(subset=columnas_features)
    test = df_completo[(df_completo["fecha_hora"] >= o["test_inicio"]) & (df_completo["fecha_hora"] <= o["test_fin"])]

    train_p = train.rename(columns={"fecha_hora": "ds", "precio_bolsa": "y"})[["ds", "y"] + columnas_regresoras_prophet].copy()
    test_p = test.rename(columns={"fecha_hora": "ds", "precio_bolsa": "y"})[["ds", "y"] + columnas_regresoras_prophet].copy()

    train_p["y"] = np.log(train_p["y"])
    train_p["precio_lag24h"] = np.log(train_p["precio_lag24h"])
    test_p["precio_lag24h"] = np.log(test_p["precio_lag24h"])

    modelo = Prophet(daily_seasonality=True, weekly_seasonality=True, yearly_seasonality=True,
                      changepoint_prior_scale=0.005)
    for col in columnas_regresoras_prophet:
        modelo.add_regressor(col)
    modelo.fit(train_p)

    pronostico = modelo.predict(test_p[["ds"] + columnas_regresoras_prophet])
    y_pred = np.exp(pronostico["yhat"].values)

    mae, rmse, mape = calcular_metricas(test["precio_bolsa"].values, y_pred)
    resultados.append({"origen": o["nombre"], "regimen": o["regimen"], "modelo": "Prophet",
                        "mae": mae, "rmse": rmse, "mape": mape})

    print(f"{o['nombre']} [{o['regimen']}]  Prophet MAE {mae:6.2f} RMSE {rmse:6.2f} MAPE {mape:5.2f}%")


C:\Users\mgdbj\xm-spot-price-predictor\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Importing plotly failed. Interactive plots will not work.


11:53:23 - cmdstanpy - INFO - Chain [1] start processing


11:53:26 - cmdstanpy - INFO - Chain [1] done processing


Origen 1 [La Nina (inicio)]  Prophet MAE  20.84 RMSE  25.61 MAPE 13.91%


...

11:54:23 - cmdstanpy - INFO - Chain [1] done processing


Origen 4 [El Nino (fuerte)]  Prophet MAE 204.69 RMSE 238.02 MAPE 42.74%


11:54:27 - cmdstanpy - INFO - Chain [1] start processing


11:54:51 - cmdstanpy - INFO - Chain [1] done processing


Origen 5 [El Nino (pico)]  Prophet MAE  52.12 RMSE  74.71 MAPE 11.23%


In [7]:
# --- Tabla consolidada ---
df_resultados = pd.DataFrame(resultados)
tabla_mae = df_resultados.pivot(index=["origen", "regimen"], columns="modelo", values="mae").round(2)
tabla_mae = tabla_mae[["Persistencia", "XGBoost", "Prophet"]]
tabla_mae["XGBoost_vs_persistencia"] = np.where(tabla_mae["XGBoost"] < tabla_mae["Persistencia"], "gana", "pierde")
tabla_mae["Prophet_vs_persistencia"] = np.where(tabla_mae["Prophet"] < tabla_mae["Persistencia"], "gana", "pierde")
tabla_mae


,modelo,Persistencia,XGBoost,Prophet,XGBoost_vs_persistencia,Prophet_vs_persistencia
origen,regimen,,,,,
Origen 1,La Nina (inicio),15.71,16.02,20.84,pierde,pierde
Origen 2,La Nina (continuacion),6.32,8.06,36.20,pierde,pierde
Origen 3,La Nina (triple-dip),36.07,42.25,38.27,pierde,pierde
Origen 4,El Nino (fuerte),90.42,129.97,204.69,pierde,pierde
Origen 5,El Nino (pico),42.49,46.96,52.12,pierde,pierde


In [8]:
# --- Estabilidad de cada modelo a lo largo de los 5 origenes/regimenes ---
# Coeficiente de variacion (desviacion estandar / media) del MAE entre origenes:
# mas alto = el modelo es menos estable frente a cambios de regimen hidrologico.
for modelo in ["Persistencia", "XGBoost", "Prophet"]:
    maes = df_resultados[df_resultados["modelo"] == modelo]["mae"]
    cv = maes.std() / maes.mean()
    print(f"{modelo:15s} MAE medio: {maes.mean():6.2f}  desv.std: {maes.std():6.2f}  CV: {cv:.2f}")

print()
ninos = df_resultados[df_resultados["regimen"].str.contains("Nino")]
ninas = df_resultados[df_resultados["regimen"].str.contains("Nina")]
for modelo in ["Persistencia", "XGBoost", "Prophet"]:
    mae_nino = ninos[ninos["modelo"] == modelo]["mae"].mean()
    mae_nina = ninas[ninas["modelo"] == modelo]["mae"].mean()
    print(f"{modelo:15s} MAE promedio El Nino: {mae_nino:6.2f}   MAE promedio La Nina: {mae_nina:6.2f}   razon: {mae_nino/mae_nina:.2f}x")


Persistencia    MAE medio:  38.20  desv.std:  32.68  CV: 0.86
XGBoost         MAE medio:  48.65  desv.std:  48.40  CV: 0.99
Prophet         MAE medio:  70.42  desv.std:  75.87  CV: 1.08

Persistencia    MAE promedio El Nino:  66.45   MAE promedio La Nina:  19.37   razon: 3.43x
XGBoost         MAE promedio El Nino:  88.46   MAE promedio La Nina:  22.11   razon: 4.00x
Prophet         MAE promedio El Nino: 128.41   MAE promedio La Nina:  31.77   razon: 4.04x


In [9]:
# --- Guardar el resultado consolidado para el informe comparativo de OE2 ---
ruta_salida = RAIZ / "data" / "processed" / "resultados" / "walkforward_5origenes.csv"
df_resultados.to_csv(ruta_salida, index=False)
print("Guardado en:", ruta_salida)


Guardado en: C:\Users\mgdbj\xm-spot-price-predictor\data\processed\resultados\walkforward_5origenes.csv
